In [5]:
import sys,requests

# part1 从世界线多维表读表，查错并录入mysql epic和epicrelationship

In [6]:
#r=requests.get('http://172.21.0.14:81/bitapp/record?bitapp_token=L5Wxb6vj3aGLcXsFKVncBle9nac&table_id=tblwqo4b4lHLzWKL')
r=requests.get('http://172.21.0.14:81/bitapp/record?app_token=L5Wxb6vj3aGLcXsFKVncBle9nac&table_id=tblwqo4b4lHLzWKL')

r

<Response [200]>

In [7]:
data=r.json()['data']

KeyError: 'data'

In [ ]:
data

[{'fields': {'下级世界线': [{'record_ids': ['recPQ1KbYX', 'rec4lXVoCO'],
     'table_id': 'tblwqo4b4lHLzWKL',
     'text': '宏观叙事,市场现象（3-12月）',
     'text_arr': ['宏观叙事', '市场现象（3-12月）'],
     'type': 'text'}],
   '世界线名称概述': '世界线',
   '开始关注时间': 1680278400000},
  'id': 'recXVxkTjv',
  'record_id': 'recXVxkTjv'},
 {'fields': {'下级世界线': [{'record_ids': ['rec5UYnVP6',
      'recofTA8R8',
      'recNxcrdrw',
      'rec3p8VJLI'],
     'table_id': 'tblwqo4b4lHLzWKL',
     'text': '逆全球化,技术进步,中国人口老龄化,财政赤字',
     'text_arr': ['逆全球化', '技术进步', '中国人口老龄化', '财政赤字'],
     'type': 'text'}],
   '世界线名称概述': '宏观叙事',
   '世界线定义': '确定性较强、信心程度较高、持续时间较长的世界线。',
   '开始关注时间': 1680278400000},
  'id': 'recPQ1KbYX',
  'record_id': 'recPQ1KbYX'},
 {'fields': {'下级世界线': [{'record_ids': ['recsROn2wu',
      'recwwjov10',
      'recQXSofE1',
      'recrmJGhUo',
      'recj2jBTKn',
      'rec5sWywyU',
      'recDrqOMqs'],
     'table_id': 'tblwqo4b4lHLzWKL',
     'text': '中国经济复苏,锂矿供给过剩-锂电池成本下降,禽流感流行,硅料大规模投产,高温导致甘蔗减产,海南封关政策,小动力钠电池替代

In [8]:
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL,get_mysql


In [9]:
mysql=get_mysql()

In [10]:
mysql.read_query('desc epic_relationship')

(('id', 'int', 'NO', 'PRI', None, 'auto_increment'),
 ('parent_id', 'int', 'YES', 'MUL', None, ''),
 ('child_id', 'int', 'YES', 'MUL', None, ''))

In [148]:
mysql.read_query('delete from  epic_relationship')
mysql.read_query('delete from  epic')

()

In [15]:
dt

datetime.datetime(2023, 4, 1, 0, 0)

In [149]:
from datetime import datetime,timedelta
def get_trend(v):
    if not v:return None
    if '↓' in v:
        return -1
    elif '—' in v:
        return 0
    elif '↑' in v:return 1
    else :return None
def refresh_epic(data):
    mysql.read_query('delete from  epic_relationship')
    mysql.read_query('delete from  epic')
    vals=[]
    children_list=[]
    for epic in data:
        '''录入状态1（没有结束关注），名字 世界线名称概述 ，介绍 世界线定义 骨干 相关骨干'''
        fields=epic['fields']
        #print(epic)
        endtime=fields.get('结束关注时间')
        if endtime:
            if  datetime.today().timestamp()*1000>endtime:
                continue
        
        children=fields.get('下级世界线')
        name=fields.get('世界线名称概述')
        if children: 
            children=children[0]['text_arr']
            for child in children:
                children_list.append((name,child))
        
        #状态 ↓—↑
        this_week=get_trend(fields.get('本周状态'))
        last_week=get_trend(fields.get('上周状态'))

        #时间为毫秒级时间戳
        info=fields.get('世界线定义')
        backbone=fields.get('相关骨干')
        val=[name if name else None,info if info else None,str(backbone[0]['text_arr'])if backbone else None,1,this_week,last_week]
        #sql=f"""insert ignore into epic (name,info,backbone,status) values ({'"'+name+'"' if name else None},{'"'+info+'"' if info else None},{'"'+str(backbone)+'"' if backbone else None},1)"""
        #print(val)
        if (children and backbone) or  ( not children and not backbone):
            print(f'error. children:{children}, backbone:{backbone}')
        #else:
        #    print((f'good. children:{children}, backbone:{backbone}'))
        vals.append(val)
    num1=mysql.write_many_query('insert ignore  into epic (name,info,backbone,status,this_week,last_week) values (%s,%s,%s,%s,%s,%s)',vals)
    for children in children_list:
        mysql.write_query(f'insert into epic_relationship (parent_id,child_id) values((select id from epic where name="{children[0]}"),(select id from epic where name="{children[1]}"))')
    return num1,len(children_list)

In [150]:
refresh_epic(data)

(99, 100)

In [28]:
#mysql.close()

# part2 从epic表找到跟某骨干相关的底层世界线并一路向上查到 宏观叙事/市场现象（3-12月） 为止，从上到下输出世界线信息 DFS?

In [5]:
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL,get_mysql
mysql=get_mysql()

In [11]:
backbone='互联网云计算'
records=mysql.read_query(f'''select id,name,info,this_week,last_week from epic where backbone like "%%'{backbone}'%%"''')
records

((1058,
  'AI+AR/VR',
  'AI技术将加速AR/VR行业的发展，例如人工智能引擎能帮助VR开发人员迅速搭建VR场景，加快VR内容制作速度。通过AI引擎，一些公司能够将设计师的设计图直接生成VR场景，并且支持六自由度的VR互动，这种方式为将现有设计资源VR化提供了大规模实践的可能性。',
  None,
  None),
 (1071,
  '基础软件国产化',
  '基础软件国产化是为了保障国家信息安全。如果一个国家的基础软件（如操作系统）依赖于外国公司，那么这个国家的信息安全就可能受到威胁。因此，许多国家都在努力发展本土的基础软件产业，以保障国家信息安全。此外，基础软件国产化还有助于促进本土IT产业的发展。通过研发和生产本土的基础软件，可以培养本土IT人才，促进本土IT企业的发展，为国家经济增长做出贡献',
  None,
  None),
 (1072,
  '政府IT建设',
  '政府IT建设是指政府在信息技术领域的投资和建设。政府IT建设和信息安全有着密切的联系。政府IT建设涉及到大量的政府数据和公民个人信息，因此信息安全至关重要。政府在IT建设过程中采用安全的技术和标准，建立健全的信息安全管理制度。',
  None,
  None),
 (1073,
  'AI自主可控',
  '逆全球化之下，叠加AI的技术突破，各国开始AI的类竞赛，因此硬件和软件作为核心原材料从而成为了核心的竞争领域，例如美国在22年9月限制英伟达出口最先进的A100，H100芯片出口中国等。',
  0,
  None),
 (1089,
  '算力紧缺（数据中心建设）',
  '随着ChatGPT等人工智能技术达到商业使用级别（成熟度提升，应用场景拓宽），产生了更多对于算力的需求。\n在模型预训练以及用户交互环节最主要消耗的原材料为算力，对此，需要建设更多的数据中心来满足日益增长的算力需求。\n其次是，研发计算能力更强的计算芯片。而数据中心的扩产难度约为18个月，同时摩尔定律逐渐失效，单位价值下算力的增长达到瓶颈，因此算力预计在未来1年-2年左右的时间内将持续短缺。',
  0,
  1),
 (1090,
  '模型能力提升',
  '生成式AI基于模型，类似于一种技术储备，下面梳理了目前能够进行模型开发的公司：\n1. OpenAI：ChatGPT\n

In [12]:
class Epic:
    def __init__(self, id,name,info,this_week,last_week,tier=None):
        self.id = id
        self.name=name
        self.info=info
        self.this_week=this_week
        self.last_week=last_week
        #self.backbone=backbone
        self.tier=tier
        self.children = []
    def __repr__(self) -> str:
        return f'{self.name}:"{self.info}"'
class EpicTree:
    def __init__(self, root=None):
        self._root = root
        self.epics={}
        self._dfs_list=[]
    @property
    def root(self):
        return self._root

    @root.setter
    def root(self, root):
        if not root in self.epics.keys():
            raise ValueError(f'this id {root} is not in this tree')
        else:
            self._root = root
    

    def __iter__(self):
       
        self.iter_index = 0
        return self

    def __next__(self):
        
        
        if not self._dfs_list:self.dfs_list(self.root)
        if self.iter_index >= len(self._dfs_list):
            raise StopIteration
        thisid = self._dfs_list[self.iter_index]
        #print(self.iter_index)
        #print(thisid)
        self.iter_index += 1
        return self.epics[thisid]
    def get_epic(self,epic_id):
        return self.epics.get(epic_id,'nothing!')
    def add_epic(self,epic):
        if epic.id not in self.epics.keys():
            self.epics[epic.id]=epic
    def add_relation(self, parent_id, child_id):
        #self.add_epic(parent)
        #self.add_epic(child)
        parent=self.epics[parent_id]
        child=self.epics[child_id]
        if child.id not in parent.children:
            parent.children.append(child.id)

    def dfs(self, node_id):
        node=self.epics[node_id]
        print(node.tier,node.name,node.info)
        for child in node.children:
            self.dfs(child)
    def dfs_list(self, node_id):
        node=self.epics[node_id]
        self._dfs_list.append(node_id)
        for child in node.children:
            self.dfs_list(child)    
    def set_tier(self,root=None,tier=0):
        
        tier=0 if not tier else tier
        try:
            node=self.epics[self.root] if not root else root
            node.tier = tier
            for child_id in node.children:
                child=self.epics[child_id]
                self.set_tier(child,tier+1)
        except:raise Exception('no root!')
    


In [15]:
def load_one_parent(child_id):
  
    
        parent=mysql.read_query(f"SELECT e.id,e.name,e.info,e.this_week,e.last_week \
            FROM epic e \
            INNER JOIN epic_relationship r ON e.id = r.parent_id \
            WHERE r.child_id = {child_id}")
        #print(parent)
        if parent:
            #print(parent)
            return parent
        else:
            
            return False

def build_tree_from_backbone(backbone='畜牧业'):
    epictree=EpicTree()
    leaves=mysql.read_query(f'''select id,name,info,this_week,last_week from epic where backbone like "%%'{backbone}'%%"''')
    for leaf in leaves:
        
        new_epic=Epic(*leaf)
        
        epictree.add_epic(new_epic)
        child_id=leaf[0]
        while True:
            get_parent=load_one_parent(child_id)
            if get_parent:
                new_epic=Epic(*get_parent[0])
                epictree.add_epic(new_epic)
                old_child_id=child_id
                child_id=get_parent[0][0]
                epictree.add_relation(child_id,old_child_id)
            else:
                if not epictree.root:
                     epictree.root=child_id
                break
    return epictree 
epictree=build_tree_from_backbone('互联网云计算')
epictree.set_tier()


In [16]:
result=''
for epic in epictree:
    if epic.tier<2:
        print(f'skip {epic.name} epic of tier{epic.tier}')
    else:
        if epic.tier==2:
            result+=str(f'{epic.tier}级世界线')+','+str(epic.name)+','+str(epic.info.replace('\n',' '))+'\n'
            print(f'{epic.tier}级世界线',epic.tier,epic.name,epic.info)
        else:
            result+=str(f'下属{epic.tier}级世界线')+','+str(epic.name)+','+str(epic.info.replace('\n',' '))+'\n'
            print(f'下属{epic.tier}级世界线',epic.tier,epic.name,epic.info)

skip 世界线 epic of tier0
skip 宏观叙事 epic of tier1
2级世界线 2 技术进步 技术进步是指生产工艺、中间投入品以及制造技能等方面的革新和改进。具体表现为对旧设备的改造和采用新设备改进旧工艺，采用新工艺使用新的原材料和能源对原有产品进行改进研究开发新产品提高工人的劳动技能等。
下属3级世界线 3 AI+AR/VR AI技术将加速AR/VR行业的发展，例如人工智能引擎能帮助VR开发人员迅速搭建VR场景，加快VR内容制作速度。通过AI引擎，一些公司能够将设计师的设计图直接生成VR场景，并且支持六自由度的VR互动，这种方式为将现有设计资源VR化提供了大规模实践的可能性。
下属3级世界线 3 AI技术突破 ChatGPT的推出为用户提供了一个参与到AI技术工作流程中的机会，也提供了一个途径以使得长期对AI应用不甚了解的用户可以切实地体会AI技术的发展。这在AI的发展历程中具有一定的里程碑意义，意味着AI技术由“幕后”转向“台前”。
下属4级世界线 4 算力紧缺（数据中心建设） 随着ChatGPT等人工智能技术达到商业使用级别（成熟度提升，应用场景拓宽），产生了更多对于算力的需求。
在模型预训练以及用户交互环节最主要消耗的原材料为算力，对此，需要建设更多的数据中心来满足日益增长的算力需求。
其次是，研发计算能力更强的计算芯片。而数据中心的扩产难度约为18个月，同时摩尔定律逐渐失效，单位价值下算力的增长达到瓶颈，因此算力预计在未来1年-2年左右的时间内将持续短缺。
下属4级世界线 4 模型能力提升 生成式AI基于模型，类似于一种技术储备，下面梳理了目前能够进行模型开发的公司：
1. OpenAI：ChatGPT
2. 百度：文心一言
3. 华为：盘古大模型
4. 阿里巴巴：通义大模型
5. 昆仑万维：天工大模型
6. 360：360智脑大模型
7. 商汤（算力储备充分）：日日新
从两个角度衡量模型能力的提升：一是能够使用的模型数量，二是模型的成熟程度。
下属4级世界线 4 数据重要性提升 Chat GPT推出预示着人工智能技术的成熟度提高了一档，各国、各公司开始人工智能竞赛，一是芯片，二是训练模型。训练模型的重要原材料为，数据。
下属4级世界线 4 人工智能下游应用 人工智能技术的提升将对下游产业产生积极影响。例如，生成式AI能够显著降低

In [31]:
result

'2级世界线,禽流感流行,美国禽流感疫情在2022年2月份就已经出现。2022年2月份之后，美国禽流感疫情持续恶化。这场危机已经蔓延至全美23个州，导致超过2700万只禽鸟被宰杀。作为此次禽流感疫情最严重的地区之一，艾奥瓦州截至目前发现12例确诊病例，其中10例来自商业养殖场，导致近1800万只禽鸟被安乐死。与此同时，随着春季鸟类迁徙，不少专家认为美国禽流感将持续加剧。\n下属3级世界线,美国种鸡供给减少,2023年春季，禽流感疫情首先在南美洲扩大传播，之后逐步扩张到环太平洋各国与欧洲地区，因疫情影响我国已将暂停禽类产品从美洲大陆各国进口。从2022年5月至今，祖代海外引种数量不超过20万套，预计从海外低引种状况或持续更长时间，种源缺口有望进一步放大。禽流感的进一步影响加剧导致日本、台湾等地区出现大量禽类被扑杀，导致局部出现“蛋荒”“鸡荒”可能进一步导致我国在禽类产品上出口结构的改变。\n下属4级世界线,中国鸡肉供给不足,各鸡苗、白羽鸡种鸡出口国暂停出口计划，导致国内鸡苗供应缺口进一步扩大。\n下属4级世界线,中国鸡蛋供应不足,周围国家、地区由于禽流感大量禽类遭到扑杀，出现“蛋荒”\n下属4级世界线,兽药、疫苗需求增加,全球禽流感影响不断扩大，导致禽类干疫苗热度升高，需求增加\n2级世界线,中国经济复苏,在过去的三年中，面对疫情反复，中国经济表现出强大韧性。2020年，中国成为全球率先实现经济正增长的主要经济体；2021年，中国经济总量占世界经济比重达18.5%，稳居世界第二位；2022年，中国经济划出一条“V”型复苏曲线，国内生产总值突破120万亿元。疫情防控措施优化调整后，中国经济预计将出现强劲复苏。\n下属3级世界线,居民消费水平恢复,疫情过后，消费需求逐渐恢复。2023年以来，中国消费市场持续回暖，重回经济增长主拉动力。中央经济工作会议将“着力扩大国内需求”确定为2023年经济工作的首要任务，明确指出“要把恢复和扩大消费摆在优先位置”，增强消费能力，改善消费条件，创新消费场景。\n下属4级世界线,猪肉、鸡肉需求提升,肉制品是我国食品结构的重要组成部分，我国是世界肉类消费大国，也是生产大国。我国肉制品行业产业链主要包括上游禽畜养殖以及饲料加工业，中游屠宰加工以及肉制品的深加工，再经过冷链物流运输等链条，肉制品最终到达消费者终端。在后疫情时代，食品安全健康的需求

In [235]:
epictree._dfs_list

[1022,
 1023,
 1026,
 1058,
 1057,
 1089,
 1090,
 1091,
 1096,
 1097,
 1098,
 1099,
 1100,
 1106,
 1025,
 1042,
 1048,
 1071,
 1072,
 1073]

In [224]:
epictree.set_tier()

In [225]:
epictree.dfs(epictree.root)

0 世界线 None
1 宏观叙事 确定性较强、信心程度较高、持续时间较长的世界线。
2 技术进步 技术进步是指生产工艺、中间投入品以及制造技能等方面的革新和改进。具体表现为对旧设备的改造和采用新设备改进旧工艺，采用新工艺使用新的原材料和能源对原有产品进行改进研究开发新产品提高工人的劳动技能等。
3 AI+AR/VR AI技术将加速AR/VR行业的发展，例如人工智能引擎能帮助VR开发人员迅速搭建VR场景，加快VR内容制作速度。通过AI引擎，一些公司能够将设计师的设计图直接生成VR场景，并且支持六自由度的VR互动，这种方式为将现有设计资源VR化提供了大规模实践的可能性。
3 AI技术突破 ChatGPT的推出为用户提供了一个参与到AI技术工作流程中的机会，也提供了一个途径以使得长期对AI应用不甚了解的用户可以切实地体会AI技术的发展。这在AI的发展历程中具有一定的里程碑意义，意味着AI技术由“幕后”转向“台前”。
4 算力紧缺（数据中心建设） 随着ChatGPT等人工智能技术达到商业使用级别（成熟度提升，应用场景拓宽），产生了更多对于算力的需求。
在模型预训练以及用户交互环节最主要消耗的原材料为算力，对此，需要建设更多的数据中心来满足日益增长的算力需求。
其次是，研发计算能力更强的计算芯片。而数据中心的扩产难度约为18个月，同时摩尔定律逐渐失效，单位价值下算力的增长达到瓶颈，因此算力预计在未来1年-2年左右的时间内将持续短缺。
4 模型能力提升 生成式AI基于模型，类似于一种技术储备，下面梳理了目前能够进行模型开发的公司：
1. OpenAI：ChatGPT
2. 百度：文心一言
3. 华为：盘古大模型
4. 阿里巴巴：通义大模型
5. 昆仑万维：天工大模型
6. 360：360智脑大模型
7. 商汤（算力储备充分）：日日新
从两个角度衡量模型能力的提升：一是能够使用的模型数量，二是模型的成熟程度。
4 数据重要性提升 Chat GPT推出预示着人工智能技术的成熟度提高了一档，各国、各公司开始人工智能竞赛，一是芯片，二是训练模型。训练模型的重要原材料为，数据。
4 人工智能下游应用 人工智能技术的提升将对下游产业产生积极影响。例如，生成式AI能够显著降低内容创作的成本，为用户提供更多便利。此外，AI与现有产品形式的融合还能够创造出新的产品形式，为用户带来更多选择。
5

In [226]:
epictree.dfs_list(epictree.root)

In [192]:
epictree.get_epic(epictree.root)

世界线:"None"

In [155]:
epictree.dfs(epictree.root)

0 世界线 None
1 宏观叙事 确定性较强、信心程度较高、持续时间较长的世界线。
2 技术进步 技术进步是指生产工艺、中间投入品以及制造技能等方面的革新和改进。具体表现为对旧设备的改造和采用新设备改进旧工艺，采用新工艺使用新的原材料和能源对原有产品进行改进研究开发新产品提高工人的劳动技能等。
3 AI+AR/VR AI技术将加速AR/VR行业的发展，例如人工智能引擎能帮助VR开发人员迅速搭建VR场景，加快VR内容制作速度。通过AI引擎，一些公司能够将设计师的设计图直接生成VR场景，并且支持六自由度的VR互动，这种方式为将现有设计资源VR化提供了大规模实践的可能性。
3 AI技术突破 ChatGPT的推出为用户提供了一个参与到AI技术工作流程中的机会，也提供了一个途径以使得长期对AI应用不甚了解的用户可以切实地体会AI技术的发展。这在AI的发展历程中具有一定的里程碑意义，意味着AI技术由“幕后”转向“台前”。
4 算力紧缺（数据中心建设） 随着ChatGPT等人工智能技术达到商业使用级别（成熟度提升，应用场景拓宽），产生了更多对于算力的需求。
在模型预训练以及用户交互环节最主要消耗的原材料为算力，对此，需要建设更多的数据中心来满足日益增长的算力需求。
其次是，研发计算能力更强的计算芯片。而数据中心的扩产难度约为18个月，同时摩尔定律逐渐失效，单位价值下算力的增长达到瓶颈，因此算力预计在未来1年-2年左右的时间内将持续短缺。
4 模型能力提升 生成式AI基于模型，类似于一种技术储备，下面梳理了目前能够进行模型开发的公司：
1. OpenAI：ChatGPT
2. 百度：文心一言
3. 华为：盘古大模型
4. 阿里巴巴：通义大模型
5. 昆仑万维：天工大模型
6. 360：360智脑大模型
7. 商汤（算力储备充分）：日日新
从两个角度衡量模型能力的提升：一是能够使用的模型数量，二是模型的成熟程度。
4 数据重要性提升 Chat GPT推出预示着人工智能技术的成熟度提高了一档，各国、各公司开始人工智能竞赛，一是芯片，二是训练模型。训练模型的重要原材料为，数据。
4 人工智能下游应用 人工智能技术的提升将对下游产业产生积极影响。例如，生成式AI能够显著降低内容创作的成本，为用户提供更多便利。此外，AI与现有产品形式的融合还能够创造出新的产品形式，为用户带来更多选择。
5

In [51]:
parents=[]
def load_one_parent(child_id):
  
    
        parent=mysql.read_query(f"SELECT e.id,e.name,e.info,e.this_week,e.last_week \
            FROM epic e \
            INNER JOIN epic_relationship r ON e.id = r.parent_id \
            WHERE r.child_id = {child_id}")
        #print(parent)
        if parent:
            print(parent)
            return parent
        else:
            
            return False
child_id=939
while True:
    child=load_one_parent(child_id)
    if child:
        child_id=child[0][0]
    else:break

((936, '美国种鸡供给减少', '2023年春季，禽流感疫情首先在南美洲扩大传播，之后逐步扩张到环太平洋各国与欧洲地区，因疫情影响我国已将暂停禽类产品从美洲大陆各国进口。从2022年5月至今，祖代海外引种数量不超过20万套，预计从海外低引种状况或持续更长时间，种源缺口有望进一步放大。禽流感的进一步影响加剧导致日本、台湾等地区出现大量禽类被扑杀，导致局部出现“蛋荒”“鸡荒”可能进一步导致我国在禽类产品上出口结构的改变。', None, None),)
((928, '禽流感流行', '美国禽流感疫情在2022年2月份就已经出现。2022年2月份之后，美国禽流感疫情持续恶化。这场危机已经蔓延至全美23个州，导致超过2700万只禽鸟被宰杀。作为此次禽流感疫情最严重的地区之一，艾奥瓦州截至目前发现12例确诊病例，其中10例来自商业养殖场，导致近1800万只禽鸟被安乐死。与此同时，随着春季鸟类迁徙，不少专家认为美国禽流感将持续加剧。', None, None),)
((922, '市场现象（3-12月）', '不太确定、预计持续时间比较短的世界线。', None, None),)
((920, '世界线', None, None, None),)


In [48]:
def get_parent(child,parents):
  
    while True:
        parent=mysql.read_query(f"SELECT e.id,e.name,e.info,e.this_week,e.last_week \
            FROM epic e \
            INNER JOIN epic_relationship r ON e.id = r.parent_id \
            WHERE r.child_id = {child}")
        if parent:
            parents.append(parent)
            id=parent[0][0]
            get_parent(id,parents)
            return parents
        else:
            return False
get_parent(939,parents)[::-1]


[((920, '世界线', None, None, None),),
 ((922, '市场现象（3-12月）', '不太确定、预计持续时间比较短的世界线。', None, None),),
 ((928,
   '禽流感流行',
   '美国禽流感疫情在2022年2月份就已经出现。2022年2月份之后，美国禽流感疫情持续恶化。这场危机已经蔓延至全美23个州，导致超过2700万只禽鸟被宰杀。作为此次禽流感疫情最严重的地区之一，艾奥瓦州截至目前发现12例确诊病例，其中10例来自商业养殖场，导致近1800万只禽鸟被安乐死。与此同时，随着春季鸟类迁徙，不少专家认为美国禽流感将持续加剧。',
   None,
   None),),
 ((936,
   '美国种鸡供给减少',
   '2023年春季，禽流感疫情首先在南美洲扩大传播，之后逐步扩张到环太平洋各国与欧洲地区，因疫情影响我国已将暂停禽类产品从美洲大陆各国进口。从2022年5月至今，祖代海外引种数量不超过20万套，预计从海外低引种状况或持续更长时间，种源缺口有望进一步放大。禽流感的进一步影响加剧导致日本、台湾等地区出现大量禽类被扑杀，导致局部出现“蛋荒”“鸡荒”可能进一步导致我国在禽类产品上出口结构的改变。',
   None,
   None),)]

In [16]:
4*4*8000

128000

In [49]:
child_id=939
mysql.read_query(f"SELECT e.id,e.name,e.info,e.this_week,e.last_week \
FROM epic e \
INNER JOIN epic_relationship r ON e.id = r.parent_id \
WHERE r.child_id = {child_id}")

((936,
  '美国种鸡供给减少',
  '2023年春季，禽流感疫情首先在南美洲扩大传播，之后逐步扩张到环太平洋各国与欧洲地区，因疫情影响我国已将暂停禽类产品从美洲大陆各国进口。从2022年5月至今，祖代海外引种数量不超过20万套，预计从海外低引种状况或持续更长时间，种源缺口有望进一步放大。禽流感的进一步影响加剧导致日本、台湾等地区出现大量禽类被扑杀，导致局部出现“蛋荒”“鸡荒”可能进一步导致我国在禽类产品上出口结构的改变。',
  None,
  None),)

# others

In [53]:
import graphviz

# 创建一个有向图
dot = graphviz.Digraph(comment='My Thoughts')

# 添加节点
for epic in epics:
    dot.node(epic[0],epic[1],fontname='SimSun')
dot.node('A', '世界线',shape='box',fontname='SimSun')
dot.node('B', '宏观叙事',fontname='SimSun')
dot.node('C', '市场现象',fontname='SimSun')
dot.node('E', '逆全球化',fontname='SimSun')
dot.node('F', '技术进步',fontname='SimSun')
dot.node('G', '中国人口老龄化',fontname='SimSun')
dot.node('H', '中国经济复苏',fontname='SimSun')
dot.node('I', '禽流感流行',fontname='SimSun')

dot.node('J', '消费电子',fontname='SimSun',shape='diamond')
dot.node('K', '旅游',fontname='SimSun',shape='diamond')
#shape: box、circle、ellipse、diamond、hexagon、octagon
# 添加边
#dot.edges(['AB', 'BC'])
dot.edge('A','B','下级世界线',fontname='SimSun')
dot.edge('A','C','下级世界线',fontname='SimSun')
dot.edge('B','E','下级世界线',fontname='SimSun')
dot.edge('B','F','下级世界线',fontname='SimSun')
dot.edge('B','G','下级世界线',fontname='SimSun')
dot.edge('C','H','下级世界线',fontname='SimSun')
dot.edge('C','I','下级世界线',fontname='SimSun')

#dot.edge('G','J','相关骨干',fontname='SimSun')
dot.edge('H','J','相关骨干',fontname='SimSun')
dot.edge('H','K','相关骨干',fontname='SimSun')
# 保存图像
#dot.attr(rankdir='Radial')
dot.attr(layout='circo')
dot.render('thoughts.gv', view=True)


'thoughts.gv.pdf'

In [13]:
epics=mysql.read_query('select id,name from epic')
relationships=mysql.read_query('select parent_id,child_id from epic_relationship')
import graphviz

# 创建一个有向图
dot = graphviz.Digraph(comment='My Thoughts')

# 添加节点
for epic in epics:
    print(epic)
    dot.node(str(epic[0]),epic[1],fontname='SimSun')
for rela in relationships:
    dot.edge(str(rela[0]),str(rela[1]),'下级世界线',fontname='SimSun')
dot.attr(layout='circo')
dot.render('thoughts.gv', view=True)


(959, 'AI+AR/VR')
(1002, 'AI+可穿戴设备')
(999, 'AI+影视')
(1008, 'AI+机器人')
(1000, 'AI+游戏')
(1001, 'AI+自动驾驶')
(958, 'AI技术突破')
(975, 'AI自主可控')
(1010, 'HJT电池量产')
(1015, 'LED技术发展')
(989, 'N型电池替代P型电池')
(1016, 'XR虚拟拍摄')
(945, '一带一路')
(996, '不动产登记')
(920, '世界线')
(925, '中国人口老龄化')
(981, '中国储能需求增加')
(985, '中国光伏组件出口量降低')
(979, '中国光伏需求增加')
(983, '中国大型储能需求增加')
(940, '中国式现代化')
(978, '中国电动车需求增加')
(926, '中国经济复苏')
(931, '中国资产价格上升（房地产）')
(949, '中国资本市场建设')
(1014, '中国降息')
(980, '中国风电需求增加')
(937, '中国鸡肉供给不足')
(938, '中国鸡蛋供应不足')
(1012, '中特估')
(998, '人工智能下游应用')
(1004, '人民币国际化')
(946, '信息安全')
(986, '储能全钒液流电池量产')
(988, '储能钠离子电池渗透率增加')
(962, '光伏电池片技术路线切换')
(969, '光刻机国产化')
(935, '全球储能系统成本下降')
(956, '全球卫星军备竞赛')
(934, '全球新能源汽车成本下降')
(942, '全球能源安全')
(948, '其他工业供应链安全')
(963, '养老保健')
(939, '兽药、疫苗需求增加')
(970, '前道检测设备国产化')
(987, '动力钠离子电池渗透率增加')
(997, '医疗消费需求恢复')
(967, '半导体划片机国产替代')
(965, '半导体设备国产化')
(976, '原油供应安全')
(1020, '国内旅游需求增加')
(944, '国防安全')
(951, '国际电动车销量增加')
(1013, '地方债务高筑')
(973, '基础软件国产化')
(921, '宏观叙事')
(933, '居民消费水平

'thoughts.gv.pdf'

In [44]:



# 创建Digraph对象，设置布局算法为radial

dot = graphviz.Digraph(comment='My Thoughts')


# 设置节点和边的属性
dot.node('A', 'Topic A')
dot.node('B', 'Subtopic B')
dot.node('C', 'Subtopic C')
dot.node('D', 'Subtopic D')
dot.node('E', 'Subtopic E')
dot.edge('A', 'B')
dot.edge('A', 'C')
dot.edge('C', 'D')
dot.edge('C', 'E')
#Layout type: "radial" not recognized. Use one of: circo dot fdp neato nop nop1 nop2 osage patchwork sfdp twopi
# 设置布局方式为Radial布局
dot.attr(layout='wopi',nodesep='5')
dot.render('thoughts1.gv', view=True)

'thoughts1.gv.pdf'